In [9]:
import itertools
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

DATA_PATH = Path("data/Q1b_NER.csv")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

MIN_SUPPORT_COUNT = 10
MAX_K = 3
TOP_N = 20

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

In [10]:
def load_recipes(path):
    """Return a Series: Recipe_ID -> sorted tuple of unique ingredient names."""
    raw = pd.read_csv(path)
    missing_cols = {"Recipe_ID", "Ingredient_Name"} - set(raw.columns)
    if missing_cols:
        raise ValueError(f"Missing columns in {path}: {missing_cols}")

    df = raw[["Recipe_ID", "Ingredient_Name"]].dropna().copy()
    df["Ingredient_Name"] = df["Ingredient_Name"].astype(str).str.strip().str.lower()
    df = df[df["Ingredient_Name"] != ""]

    recipes = (df.groupby("Recipe_ID", sort=True)["Ingredient_Name"]
                 .agg(lambda s: tuple(sorted(set(s)))))
    recipes = recipes[recipes.map(len) > 0]

    report = {
        "raw rows": len(raw),
        "rows with a valid ingredient name": len(df),
        "repeated (recipe, ingredient) rows collapsed": int(df.duplicated(["Recipe_ID", "Ingredient_Name"]).sum()),
        "recipes (transactions)": len(recipes),
        "ingredient occurrences after de-duplication": int(recipes.map(len).sum()),
        "unique ingredients (items)": len({i for r in recipes for i in r}),
    }
    return recipes, report


recipes, report = load_recipes(DATA_PATH)
for k, v in report.items():
    print(f"{k:<46}: {v:,}")

sizes = recipes.map(len)
print("\nRecipe size (unique ingredients per recipe):")
print(sizes.describe().round(2).to_string())

transactions = list(recipes.values)
N = len(transactions)
assert N > 0, "No recipes loaded"

raw rows                                      : 102,191
rows with a valid ingredient name             : 102,191
repeated (recipe, ingredient) rows collapsed  : 2,317
recipes (transactions)                        : 9,988
ingredient occurrences after de-duplication   : 99,874
unique ingredients (items)                    : 12,641

Recipe size (unique ingredients per recipe):
count    9988.0
mean       10.0
std         4.2
min         1.0
25%         7.0
50%        10.0
75%        13.0
max        33.0


In [11]:
def count_candidates(transactions, candidates, k, allowed_items=None):
    """Support counting pass: one scan over all recipes.

    For every recipe, enumerate its k-subsets (only over frequent single items)
    and increment the count of those that are candidates.
    """
    counts = dict.fromkeys(candidates, 0)
    for t in transactions:
        if allowed_items is not None:
            t = tuple(i for i in t if i in allowed_items)
        if len(t) < k:
            continue
        for combo in itertools.combinations(t, k):
            if combo in counts:
                counts[combo] += 1
    return counts


def apriori_gen(frequent_prev, k):
    """Join + prune step: build size-k candidates from frequent (k-1)-itemsets."""
    prev_set = set(frequent_prev)
    by_prefix = defaultdict(list)
    for itemset in frequent_prev:
        by_prefix[itemset[:-1]].append(itemset[-1])

    candidates = set()
    for prefix, last_items in by_prefix.items():
        for a, b in itertools.combinations(sorted(last_items), 2):
            cand = prefix + (a, b)
            if all(sub in prev_set for sub in itertools.combinations(cand, k - 1)):
                candidates.add(cand)
    return candidates


def apriori(transactions, min_count, max_k):
    """Return {k: {itemset_tuple: support_count}} for all frequent itemsets up to max_k."""
    c1 = Counter(i for t in transactions for i in t)
    frequent = {1: {(i,): c for i, c in c1.items() if c >= min_count}}
    frequent_items = {i for (i,) in frequent[1]}
    print(f"k=1: {len(c1):,} candidates -> {len(frequent[1]):,} frequent")

    for k in range(2, max_k + 1):
        candidates = apriori_gen(sorted(frequent[k - 1]), k)
        counts = count_candidates(transactions, candidates, k, frequent_items)
        frequent[k] = {s: c for s, c in counts.items() if c >= min_count}
        print(f"k={k}: {len(candidates):,} candidates -> {len(frequent[k]):,} frequent")
        if not frequent[k]:
            break
    return frequent


frequent = apriori(transactions, MIN_SUPPORT_COUNT, MAX_K)

k=1: 12,641 candidates -> 988 frequent
k=2: 487,578 candidates -> 6,479 frequent
k=3: 58,175 candidates -> 8,054 frequent


In [12]:
def to_dataframe(itemset_counts, k, n_transactions):
    rows = []
    for itemset, count in itemset_counts.items():
        row = {"Itemset": " | ".join(itemset), "Size": k}
        row.update({f"Item_{j + 1}": item for j, item in enumerate(itemset)})
        row["Support_Count"] = count
        row["Support"] = count / n_transactions
        rows.append(row)
    cols = ["Itemset", "Size"] + [f"Item_{j + 1}" for j in range(k)] + ["Support_Count", "Support"]
    df = pd.DataFrame(rows, columns=cols)
    df = df.sort_values(["Support_Count", "Itemset"], ascending=[False, True], kind="mergesort")
    df.insert(0, "Rank", range(1, len(df) + 1))
    return df.reset_index(drop=True)


itemset_tables = {}
for k in range(1, MAX_K + 1):
    df_k = to_dataframe(frequent.get(k, {}), k, N)
    itemset_tables[k] = df_k
    path = OUT_DIR / f"Q1a_itemsets_size_{k}.csv"
    df_k.to_csv(path, index=False)
    print(f"Size {k}: {len(df_k):>6,} frequent itemsets -> {path}")

itemset_tables[2].head()

Size 1:    988 frequent itemsets -> outputs/Q1a_itemsets_size_1.csv
Size 2:  6,479 frequent itemsets -> outputs/Q1a_itemsets_size_2.csv
Size 3:  8,054 frequent itemsets -> outputs/Q1a_itemsets_size_3.csv


,Rank,Itemset,Size,Item_1,Item_2,Support_Count,Support
0,1,garlic | olive oil,2,garlic,olive oil,1217,0.121846
1,2,garlic | onion,2,garlic,onion,789,0.078995
2,3,garlic | lemon,2,garlic,lemon,622,0.062275
3,4,butter | plain flour,2,butter,plain flour,594,0.059471
4,5,eggs | plain flour,2,eggs,plain flour,568,0.056868


In [13]:
tidsets = defaultdict(set)
for idx, t in enumerate(transactions):
    for item in t:
        tidsets[item].add(idx)

frequent_items = set(itemset_tables[1]["Item_1"])
for k in range(1, MAX_K + 1):
    df_k = itemset_tables[k]
    item_cols = [f"Item_{j + 1}" for j in range(k)]
    itemsets = [tuple(r) for r in df_k[item_cols].itertuples(index=False)]

    assert (df_k["Size"] == k).all()
    assert all(len(set(s)) == k for s in itemsets)
    assert len(set(itemsets)) == len(itemsets)
    assert (df_k["Support_Count"] >= MIN_SUPPORT_COUNT).all()
    assert (abs(df_k["Support"] - df_k["Support_Count"] / N) < 1e-12).all()
    recount = [len(set.intersection(*(tidsets[i] for i in s))) for s in itemsets]
    assert recount == df_k["Support_Count"].tolist()
    brute = Counter()
    for t in transactions:
        brute.update(itertools.combinations([i for i in t if i in frequent_items], k))
    brute_frequent = {s for s, c in brute.items() if c >= MIN_SUPPORT_COUNT}
    assert brute_frequent == set(itemsets), f"size {k}: Apriori result incomplete"
    assert len(pd.read_csv(OUT_DIR / f"Q1a_itemsets_size_{k}.csv")) == len(df_k)
    print(f"Size {k}: all checks passed ({len(df_k):,} itemsets)")

Size 1: all checks passed (988 itemsets)
Size 2: all checks passed (6,479 itemsets)
Size 3: all checks passed (8,054 itemsets)


In [14]:
sens_rows = []
for t in [5, 10, 20, 50, 100]:
    f = apriori(transactions, t, MAX_K) if t != MIN_SUPPORT_COUNT else frequent
    sens_rows.append({"Min support (recipes)": t, "Min support (%)": round(100 * t / N, 3),
                      **{f"# size-{k}": len(f.get(k, {})) for k in range(1, MAX_K + 1)}})
pd.DataFrame(sens_rows)

k=1: 12,641 candidates -> 1,738 frequent
k=2: 1,509,453 candidates -> 15,316 frequent
k=3: 205,678 candidates -> 27,367 frequent
k=1: 12,641 candidates -> 611 frequent
k=2: 186,355 candidates -> 2,678 frequent
k=3: 15,315 candidates -> 2,347 frequent
k=1: 12,641 candidates -> 291 frequent
k=2: 42,195 candidates -> 755 frequent
k=3: 2,253 candidates -> 426 frequent
k=1: 12,641 candidates -> 167 frequent
k=2: 13,861 candidates -> 253 frequent
k=3: 442 candidates -> 93 frequent


,Min support (recipes),Min support (%),# size-1,# size-2,# size-3
0,5,0.050,1738,15316,27367
1,10,0.100,988,6479,8054
2,20,0.200,611,2678,2347
3,50,0.501,291,755,426
4,100,1.001,167,253,93


In [15]:
top_tables = {}
for k in range(1, MAX_K + 1):
    df_k = itemset_tables[k]
    if len(df_k) < TOP_N:
        print(f"Note: only {len(df_k)} frequent itemsets of size {k}; exporting all of them.")
    top = df_k.head(TOP_N)[["Rank", "Itemset", "Size", "Support_Count", "Support"]].copy()
    top_tables[k] = top

    assert top["Support_Count"].is_monotonic_decreasing
    top.to_csv(OUT_DIR / f"Q1b_top20_size_{k}.csv", index=False)

    print(f"\n=== Top {len(top)} itemsets of size {k} (descending support) ===")
    print(top.assign(Support=top["Support"].map("{:.4f}".format)).to_string(index=False))


=== Top 20 itemsets of size 1 (descending support) ===
 Rank             Itemset  Size  Support_Count Support
    1              garlic     1           3104  0.3108
    2           olive oil     1           2141  0.2144
    3              butter     1           1817  0.1819
    4         plain flour     1           1643  0.1645
    5                eggs     1           1589  0.1591
    6               lemon     1           1357  0.1359
    7               onion     1           1152  0.1153
    8 golden caster sugar     1            962  0.0963
    9        caster sugar     1            942  0.0943
   10       vegetable oil     1            882  0.0883
   11        double cream     1            873  0.0874
   12                 egg     1            793  0.0794
   13         icing sugar     1            777  0.0778
   14                milk     1            770  0.0771
   15              ginger     1            743  0.0744
   16     vanilla extract     1            724  0.0725
   17    

## Q1(c) — Support and Confidence in itemset mining

Let the cuisine be a set of **N transactions** (recipes), each a set of **items** (ingredients). Let X and Y be itemsets with no ingredient in common.

### Support
$$\text{Support}(X) = \frac{\text{number of recipes containing every ingredient in } X}{N}$$

1. Support measures **how common** an ingredient combination is in the cuisine. It is a fraction between 0 and 1. The absolute version is the *support count*, e.g. Support_Count({garlic, olive oil}) = number of recipes using both.
2. Support is used as the **threshold that defines a frequent itemset**: X is frequent if Support(X) ≥ min-support.
3. Support is **anti-monotone**: adding an ingredient can never increase support (Support({a,b,c}) ≤ Support({a,b})). This is the property Apriori uses to prune candidates.

### Confidence
$$\text{Confidence}(X \Rightarrow Y) = \frac{\text{Support}(X \cup Y)}{\text{Support}(X)} = P(Y \mid X)$$

1. Confidence is defined for an **association rule** X ⇒ Y (antecedent ⇒ consequent), not for an itemset. It estimates the conditional probability that a recipe contains Y **given** that it contains X.
2. Confidence is **directional**: Confidence(X ⇒ Y) ≠ Confidence(Y ⇒ X) in general, because the denominators differ.
3. A rule is reported only if it passes both a minimum support and a minimum confidence. High confidence means co-occurrence, **not causation**. Also, a rule whose consequent is very common (e.g. ⇒ garlic) can have high confidence just because Y is common. *Lift* = Confidence / Support(Y) corrects for this.

### FISM vs. association-rule mining
Frequent itemset mining finds frequent ingredient **combinations** using support alone; this is what Q1(a) and Q1(b) do. Association-rule mining is the next step: it splits each frequent itemset into rules X ⇒ Y and scores them with confidence.

**Hypothetical example (not from our data):** in 1,000 recipes, 200 contain *flour*, 120 contain both *flour* and *butter*.
Support({flour, butter}) = 120/1000 = 0.12. Confidence(flour ⇒ butter) = 0.12 / 0.20 = 0.60. In words, 60 % of the hypothetical flour recipes also use butter.

In [16]:
support_1 = dict(zip(itemset_tables[1]["Item_1"], itemset_tables[1]["Support"]))
rule_rows = []
for _, r in itemset_tables[2].head(5).iterrows():
    a, b, s_ab = r["Item_1"], r["Item_2"], r["Support"]
    for x, y in [(a, b), (b, a)]:
        conf = s_ab / support_1[x]
        rule_rows.append({"Rule (X => Y)": f"{x} => {y}", "Support(X u Y)": round(s_ab, 4),
                          "Support(X)": round(support_1[x], 4), "Confidence": round(conf, 3),
                          "Lift": round(conf / support_1[y], 2)})
pd.DataFrame(rule_rows)

,Rule (X => Y),Support(X u Y),Support(X),Confidence,Lift
0,garlic => olive oil,0.1218,0.3108,0.392,1.83
1,olive oil => garlic,0.1218,0.2144,0.568,1.83
2,garlic => onion,0.0790,0.3108,0.254,2.20
3,onion => garlic,0.0790,0.1153,0.685,2.20
4,garlic => lemon,0.0623,0.3108,0.200,1.47
5,lemon => garlic,0.0623,0.1359,0.458,1.47
6,butter => plain flour,0.0595,0.1819,0.327,1.99
7,plain flour => butter,0.0595,0.1645,0.362,1.99
8,eggs => plain flour,0.0569,0.1591,0.357,2.17
9,plain flour => eggs,0.0569,0.1645,0.346,2.17
